In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
# import sys
# sys.path.append("..")
from Wiki import semantic_search_milvus_wiki



NameError: name 'true' is not defined

In [ ]:
load_dotenv()
client = OpenAI()

def build_context(results):

    context = ""
    sources = []

    for r in results:
        chunk = r["text"]
        source = r["url"]

        if len(context) + len(chunk) > 3000:  # Kontextlänge begrenzen
            break

        context += chunk + "\n\n"
        sources.append(source)
    return context.strip(), sources

In [ ]:



def generate_answer(query: str, k: int = 5):
    """
    Vollständige RAG-Pipeline:
    Retrieval → Prompt → LLM → Antwort
    """

    # 1. Retrieval aus Milvus
    results = semantic_search_milvus_wiki(query, k=k)

    if not results:
        return {
            "answer": "Keine passenden Informationen gefunden.",
            "sources": []
        }

    # 2. Kontext bauen
    context, sources = build_context(results)

    # 3. Prompt
    prompt = f"""
Du bist ein Reiseassistent für das Region Spreewald.

Beantworte die Frage nur anhand des gegebenen Kontexts.
Wenn du keine Antwort auf die Frage findest, sagt das ehrlich. Du darfst nicht fantasieren oder
nicht existierende Orte, Preise oder Öffnungszeiten nennen. Du musst immer die Quellen angeben, auf
denen deine Antwort basiert. Du darfst auch die Quellen nicht erfinden.

Kontext:
{context}

Frage:
{query}

Antwort:
"""

    # 4. LLM aufrufen
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Du bist ein hilfreicher Assistent."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )

    answer = response.choices[0].message.content

    return {
        "answer": answer,
        "sources": list(set(sources))  # Duplikate entfernen
    }

In [ ]:
# TEST
#================================

query = "Was ist der Spreewald und welche Geschichte hat er?"

result = generate_answer(query)

print("\n🧠 Antwort:\n")
print(result["answer"])

print("\n📚 Quellen:\n")
for s in result["sources"]:
    print("-", s)